# 06 — Rule families

Translate forecast budgets into realised risk, compare matched trend families, and screen optional native rules without a new research framework.

## Native risk, Donchian, and optional families

This is a deliberately linear teaching experiment.  It answers three related
questions without adding another research framework:

1. What does a 70% carry / 30% trend **forecast budget** become in realised
   portfolio volatility?
2. Does repository-native, continuous Donchian (`breakout`) beat buffered
   EWMAC when the speed grid and every portfolio control are matched?
3. Which of the 28 optional rules improve a carry/EWMAC core when we change
   one family at a time?

All system construction is visible below.  It uses only stable repository
stages plus the small point-in-time portfolio gate already used by notebooks
9, 10, and 18.  There is no imported research `system.py`, inherited YAML, or
new helper module.

## Frozen protocol

- Data: the 95 reviewed Chinese futures in `dbFuturesSimData`, clipped through
  2026-07-27; CNH/Asia metadata is asserted.
- Core: four native carry horizons and three medium/slow EWMAC horizons.
- Common controls: 100m CNH fixed capital, 16% target, equal weight across
  causally eligible markets, IDM 2.5, fixed source scalars, 10% forecast
  buffer, delayed whole-contract fills, and stored cash/spread costs.
- Risk attribution is descriptive.  It reports native weighted-rule gross
  attribution and a separately costed net-sleeve approximation, then measures
  the residual against the actual buffered portfolio.
- Continuous-Donchian comparison: fixed EWMAC 16/64, 32/128, 64/256 versus
  fixed breakout 40, 80, 160.  Both use equal weights, FDM 1.5, the same
  readiness panel, and the same buffer.  No lookback is fitted.
- Optional-family screen: fit only from 2012-07-28 through 2023-07-27.  Every
  non-momentum candidate receives one fixed 10% sleeve; the remaining 90%
  preserves the 70/30 carry/EWMAC ratio.  Every canonical horizon in that
  family stays.  `momentum16/32/64` are identical to the three core EWMAC
  signals, so momentum is tested honestly as a five-speed **replacement**.
- A family passes only if pre-2023 net Sharpe improves by at least 0.05, the
  median annual-block Sharpe change is positive, at least two-thirds of valid
  blocks improve, total net return improves, and the 2x-cost Sharpe does not
  deteriorate.  These are screening rules, not statistical proof.
- 2023-07-28 through 2026-07-27 is already revealed.  It never votes in the
  screen.  The frozen screened combination is then relaunched flat for that
  period and reported unchanged.

The native broad `AssetClass` grouping is used for the first-pass
cross-sectional rules.  The data object contains only Chinese instruments, so
no international market enters a forecast.  Earlier v00/v01 diagnostics found
the broad-versus-China taxonomy change immaterial at total-system level; a
cross-sectional family that passes here still needs the stricter China-peer
robustness check before production.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import research as R

R.set_notebook_style()

In [ ]:
import gc
import math
from IPython.utils.io import capture_output

from sysdata.config.configdata import Config
from sysdata.sim.db_futures_sim_data import dbFuturesSimData
from sysobjects.adjusted_prices import futuresAdjustedPrices
from sysobjects.multiple_prices import futuresMultiplePrices
from sysobjects.spot_fx_prices import fxPrices
from systems.accounts.accounts_stage import Account
from systems.basesystem import System
from systems.forecast_combine import ForecastCombine
from systems.forecast_scale_cap import ForecastScaleCap
from systems.forecasting import Rules
from systems.positionsizing import PositionSizing
from systems.provided.rob_system.rawdata import myFuturesRawData as RobRawData

R.limit_blas_threads()

POST_2008 = pd.Timestamp("2008-07-28")
FAMILY_FIT_START = pd.Timestamp("2012-07-28")
FIT_END = pd.Timestamp("2023-07-27")
AUDIT_START = pd.Timestamp("2023-07-28")
CUTOFF = pd.Timestamp("2026-07-27")
TRADING_DAYS = 256.0
CAPITAL = 100_000_000
TARGET_VOL = 16.0

## 1. The database boundary

For research, a cutoff must live in the data object rather than in the last
line of a plot.  The subclass below clips adjusted prices, multiple prices,
and FX inclusively.  Adjusted prices are additive Panama levels: native rules
use their differences; this notebook never calls percentage change on them.

In [ ]:
class CutoffChinaData(dbFuturesSimData):
    def __init__(self, cutoff):
        self.cutoff = (
            pd.Timestamp(cutoff).normalize()
            + pd.Timedelta(days=1)
            - pd.Timedelta(nanoseconds=1)
        )
        manifest = R.TushareInstrumentManifest.from_csv()
        expected = sorted(
            item.instrument_code
            for item in manifest.mappings
            if item.is_stitchable
        )
        assert len(expected) == len(set(expected)) == 95
        self._instruments = tuple(expected)
        super().__init__()

        stored = set(super().get_instrument_list())
        missing = sorted(set(expected) - stored)
        if missing:
            raise ValueError(f"Database is missing reviewed instruments: {missing}")

        metadata = self.get_all_instrument_data_as_df().reindex(expected)
        bad = metadata.index[
            (metadata["Currency"] != "CNH") | (metadata["Region"] != "ASIA")
        ].tolist()
        if bad:
            raise ValueError(f"Non-Chinese instruments exposed: {bad}")

        listed = []
        for instrument in expected:
            raw = self.db_futures_adjusted_prices_data.get_adjusted_prices(
                instrument
            ).dropna()
            if len(raw) and raw.index[0] <= self.cutoff:
                listed.append(instrument)
        self._instruments = tuple(listed)

    def get_instrument_list(self):
        return list(self._instruments)

    def _check(self, instrument):
        if instrument not in self._instruments:
            raise ValueError(f"{instrument} is outside this cutoff universe")

    def get_backadjusted_futures_price(self, instrument_code):
        self._check(instrument_code)
        prices = super().get_backadjusted_futures_price(instrument_code)
        return futuresAdjustedPrices(pd.Series(prices.loc[: self.cutoff]).copy())

    def get_multiple_prices_from_start_date(self, instrument_code, start_date):
        self._check(instrument_code)
        prices = super().get_multiple_prices_from_start_date(
            instrument_code, start_date=start_date
        )
        return futuresMultiplePrices(pd.DataFrame(prices.loc[: self.cutoff]).copy())

    def _get_fx_data_from_start_date(self, currency1, currency2, start_date):
        prices = super()._get_fx_data_from_start_date(
            currency1, currency2, start_date=start_date
        )
        return fxPrices(pd.Series(prices.loc[: self.cutoff]).copy())


data = CutoffChinaData(CUTOFF)
ALL = R.chinese_universe(data)
assert len(ALL) == 95
assert max(data.daily_prices(code).index.max() for code in ALL) <= data.cutoff
print(f"{len(ALL)} Chinese instruments; cutoff {CUTOFF.date()}")

## 2. The rules are plain dictionaries

This is the simplest way to register a rule in pysystemtrade: give `Rules` a
function path, the native stage data it consumes, and its arguments.  Scalars
below are the fixed values in `systems/provided/rob_system/config.yaml`; they
normalise forecast amplitude, not expected return.

The 28 optional names are kept in seven economic families.  We do not choose
the winning horizon after seeing Chinese returns.

In [ ]:
EWMAC_FUNCTION = "systems.provided.rules.ewmac.ewmac"
EWMAC_DATA = ["rawdata.get_daily_prices", "rawdata.daily_returns_volatility"]
CARRY_FUNCTION = "systems.provided.rules.carry.carry"
BREAKOUT_FUNCTION = "systems.provided.rules.breakout.breakout"

CARRY_RULES = ("carry10", "carry30", "carry60", "carry125")
TREND_RULES = ("ewmac16_64", "ewmac32_128", "ewmac64_256")
CORE_RULES = CARRY_RULES + TREND_RULES

TRADING_RULES = {
    "carry10": dict(function=CARRY_FUNCTION, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=10)),
    "carry30": dict(function=CARRY_FUNCTION, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=30)),
    "carry60": dict(function=CARRY_FUNCTION, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=60)),
    "carry125": dict(function=CARRY_FUNCTION, data=["rawdata.raw_carry"],
                     other_args=dict(smooth_days=125)),
    "ewmac16_64": dict(function=EWMAC_FUNCTION, data=EWMAC_DATA,
                       other_args=dict(Lfast=16, Lslow=64)),
    "ewmac32_128": dict(function=EWMAC_FUNCTION, data=EWMAC_DATA,
                        other_args=dict(Lfast=32, Lslow=128)),
    "ewmac64_256": dict(function=EWMAC_FUNCTION, data=EWMAC_DATA,
                        other_args=dict(Lfast=64, Lslow=256)),
}

for lookback in (10, 20, 40, 80, 160, 320):
    TRADING_RULES[f"breakout{lookback}"] = dict(
        function=BREAKOUT_FUNCTION,
        data=["rawdata.get_daily_prices"],
        other_args=dict(lookback=lookback),
    )
for fast in (4, 8, 16, 32, 64):
    TRADING_RULES[f"momentum{fast}"] = dict(
        function=EWMAC_FUNCTION,
        data=EWMAC_DATA,
        other_args=dict(Lfast=fast, Lslow=4 * fast),
    )
for fast in (16, 32, 64):
    TRADING_RULES[f"accel{fast}"] = dict(
        function="systems.provided.rules.accel.accel",
        data=EWMAC_DATA,
        other_args=dict(Lfast=fast),
    )
for fast in (2, 4, 8, 16, 32, 64):
    TRADING_RULES[f"assettrend{fast}"] = dict(
        function="systems.provided.rules.ewmac.ewmac_calc_vol",
        data=["rawdata.normalised_price_for_asset_class"],
        other_args=dict(Lfast=fast, Lslow=4 * fast),
    )
for horizon in (10, 20, 40, 80):
    TRADING_RULES[f"relmomentum{horizon}"] = dict(
        function="systems.provided.rules.rel_mom.relative_momentum",
        data=[
            "rawdata.get_cumulative_daily_vol_normalised_returns",
            "rawdata.normalised_price_for_asset_class",
        ],
        other_args=dict(horizon=horizon),
    )
for lookback, smooth in ((180, 45), (365, 90)):
    common = dict(
        function="systems.provided.rules.factors.factor_trading_rule",
        data=["rawdata.get_demeanded_factor_value"],
    )
    TRADING_RULES[f"skewabs{lookback}"] = dict(
        **common,
        other_args=dict(
            smooth=smooth,
            _factor_name="neg_skew",
            _demean_method="historic_average_factor_value_all_assets",
            _lookback_days=lookback,
        ),
    )
    TRADING_RULES[f"skewrv{lookback}"] = dict(
        **common,
        other_args=dict(
            smooth=smooth,
            _factor_name="neg_skew",
            _demean_method="average_factor_value_in_asset_class_for_instrument",
            _lookback_days=lookback,
        ),
    )

RULE_FAMILIES = {
    "momentum": tuple(f"momentum{x}" for x in (4, 8, 16, 32, 64)),
    "accel": tuple(f"accel{x}" for x in (16, 32, 64)),
    "assettrend": tuple(f"assettrend{x}" for x in (2, 4, 8, 16, 32, 64)),
    "breakout": tuple(f"breakout{x}" for x in (10, 20, 40, 80, 160, 320)),
    "relmomentum": tuple(f"relmomentum{x}" for x in (10, 20, 40, 80)),
    "skewabs": ("skewabs180", "skewabs365"),
    "skewrv": ("skewrv180", "skewrv365"),
}
OPTIONAL_RULES = tuple(
    rule for family in RULE_FAMILIES.values() for rule in family
)
assert len(OPTIONAL_RULES) == 28

FORECAST_SCALARS = {
    "carry10": 27.815707053556984,
    "carry30": 28.384062881349813,
    "carry60": 28.40072429176199,
    "carry125": 29.366474500729886,
    "ewmac16_64": 3.75,
    "ewmac32_128": 2.65,
    "ewmac64_256": 1.87,
    "momentum4": 8.539940954709955,
    "momentum8": 5.949404365193165,
    "momentum16": 4.104172020369661,
    "momentum32": 2.786994330124792,
    "momentum64": 1.9093945630747895,
    "accel16": 7.8170710605387095,
    "accel32": 5.563487137713779,
    "accel64": 3.896720541225276,
    "assettrend2": 10.846520114531351,
    "assettrend4": 7.572334583056326,
    "assettrend8": 5.190470936448635,
    "assettrend16": 3.549452858682833,
    "assettrend32": 2.3449234496490723,
    "assettrend64": 1.5465144366886119,
    "breakout10": 0.6031025130185256,
    "breakout20": 0.6742627921625178,
    "breakout40": 0.7036929411910525,
    "breakout80": 0.726260784624834,
    "breakout160": 0.7388310187414805,
    "breakout320": 0.7366197028421859,
    "relmomentum10": 61.24026078373817,
    "relmomentum20": 86.50746400987076,
    "relmomentum40": 117.77937298659975,
    "relmomentum80": 159.87802982511536,
    "skewabs180": 4.590246757939031,
    "skewabs365": 2.351483885205172,
    "skewrv180": 5.244752769697409,
    "skewrv365": 3.002222097593425,
}
assert set(CORE_RULES + OPTIONAL_RULES) == set(FORECAST_SCALARS)

duplicate_map = {
    "momentum16": "ewmac16_64",
    "momentum32": "ewmac32_128",
    "momentum64": "ewmac64_256",
}
for momentum, core_name in duplicate_map.items():
    assert TRADING_RULES[momentum]["other_args"] == TRADING_RULES[core_name]["other_args"]

family_table = pd.DataFrame({
    family: pd.Series(rules) for family, rules in RULE_FAMILIES.items()
})
display(family_table)
print("The three duplicate momentum/core pairs are:", duplicate_map)

## 3. One readable native system factory

The only parameters that vary between experiments are the active rule names,
their forecast weights, FDM, and the already-computed daily eligibility and
instrument weights.  Every other control is locked here.

`RobRawData` is the repository stage required by the native skew and
cross-sectional rules.  `R.PointInTimePortfolios` only gates and renormalises
instrument weights; forecasts, volatility, sizing, buffering, fills, and P&L
remain native.

In [ ]:
def core_weights(carry_budget=0.70):
    return {
        **{rule: carry_budget / len(CARRY_RULES) for rule in CARRY_RULES},
        **{
            rule: (1.0 - carry_budget) / len(TREND_RULES)
            for rule in TREND_RULES
        },
    }


def equal_weights(rule_names):
    return {rule: 1.0 / len(rule_names) for rule in rule_names}


def native_system(rule_names, forecast_weights, eligibility, fixed_weights,
                  fdm=1.0):
    rule_names = tuple(rule_names)
    assert set(rule_names) == set(forecast_weights)
    assert abs(sum(forecast_weights.values()) - 1.0) < 1e-12
    config = Config(dict(
        trading_rules={rule: TRADING_RULES[rule] for rule in rule_names},
        forecast_scalars={rule: FORECAST_SCALARS[rule] for rule in rule_names},
        forecast_weights=forecast_weights,
        use_forecast_scale_estimates=False,
        use_forecast_weight_estimates=False,
        forecast_div_multiplier=fdm,
        use_forecast_div_mult_estimates=False,
        forecast_weight_ewma_span=1,
        forecast_cap=20.0,
        average_absolute_forecast=10.0,
        instruments=ALL,
        instrument_weights={name: 1 / len(ALL) for name in ALL},
        instrument_div_multiplier=2.5,
        use_instrument_weight_estimates=False,
        use_instrument_div_mult_estimates=False,
        instrument_weight_ewma_span=1,
        notional_trading_capital=CAPITAL,
        percentage_vol_target=TARGET_VOL,
        base_currency="CNH",
        capital_multiplier=dict(func="syscore.capital.fixed_capital"),
        buffer_method="forecast",
        buffer_size=0.10,
        buffer_trade_to_edge=True,
        use_SR_costs=False,
        forecast_post_ceiling_cost_SR=999.0,
        vol_normalise_currency_costs=False,
        multiply_roll_costs_by=0.5,
        volatility_calculation=dict(
            func="sysquant.estimators.vol.mixed_vol_calc",
            name_returns_attr_in_rawdata="daily_returns",
            multiplier_to_get_daily_vol=1.0,
            days=35,
            min_periods=10,
            slow_vol_years=20,
            proportion_of_slow_vol=0.35,
            vol_abs_min=0.0000000001,
            backfill=False,
        ),
    ))
    return System(
        [Account(), R.PointInTimePortfolios(eligibility, fixed_weights),
         PositionSizing(), RobRawData(), ForecastCombine(),
         ForecastScaleCap(), Rules()],
        data,
        config,
    )


def accounting_frame(curve):
    frame = pd.concat({
        "gross": curve.percent.gross.as_ts,
        "costs": curve.percent.costs.as_ts,
        "net": curve.percent.as_ts,
    }, axis=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    assert (frame["net"] - frame["gross"] - frame["costs"]).abs().max() < 1e-8
    return frame


def portfolio_frame(system):
    curve = system.accounts.portfolio(delayfill=True, roundpositions=True)
    return accounting_frame(curve)


def sharpe(returns):
    clean = returns.replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    volatility = clean.std(ddof=1)
    if len(clean) < 2 or not np.isfinite(volatility) or volatility <= 0:
        return np.nan
    return clean.mean() / volatility * math.sqrt(TRADING_DAYS)


def compounded_stats(returns, scale=1.0):
    daily = returns.dropna().astype(float) * scale / 100.0
    wealth = (1.0 + daily).cumprod()
    anchor = pd.Series(
        [1.0], index=[daily.index[0] - pd.Timedelta(nanoseconds=1)]
    )
    wealth = pd.concat([anchor, wealth])
    drawdown = wealth / wealth.cummax() - 1.0
    years = (daily.index[-1] - daily.index[0]).days / 365.25
    ending = wealth.iloc[-1]
    cagr = ending ** (1.0 / years) - 1.0 if ending > 0 and years > 0 else np.nan
    return 100 * cagr, 100 * drawdown.min()


def performance(frame, start, end):
    sample = frame.loc[start:end]
    net = sample["net"]
    ann_vol = net.std(ddof=1) * math.sqrt(TRADING_DAYS)
    cagr, drawdown = compounded_stats(net)
    common_scale = TARGET_VOL / ann_vol if ann_vol > 0 else np.nan
    _, common_drawdown = compounded_stats(net, common_scale)
    return {
        "observations": len(sample),
        "net return %": net.sum(),
        "net Sharpe": sharpe(net),
        "2x-cost Sharpe": sharpe(sample["gross"] + 2 * sample["costs"]),
        "ann vol %": ann_vol,
        "cost drag %": -sample["costs"].sum(),
        "full CAGR %": cagr,
        "full max DD %": drawdown,
        "16%-vol max DD %": common_drawdown,
    }

## 4. One causal membership panel, many rule-specific readiness masks

Liquidity enters at a trailing 20-observed-session mean of 130 contracts and
exits below 70.  A rule is eligible only after all rules active in that exact
comparison and native volatility are ready.  This prevents pandas' all-NaN
sum from becoming a plausible zero forecast.

Native breakout currently chooses its rolling `min_periods` from the final
length of the supplied series.  We add the economically natural half-window
observation gate, making a short cutoff agree with the prefix of a later
cutoff without modifying the user's `breakout.py`.

In [ ]:
print("reading held-contract volume ...")
with capture_output():
    held_volume = R.held_contract_volumes(data, ALL)
liquidity = R.liquidity_eligibility(held_volume, force_terminal_close=True)

print("building all 35 native forecasts once for readiness ...")
PROBE_RULES = CORE_RULES + OPTIONAL_RULES
probe = native_system(
    PROBE_RULES,
    equal_weights(PROBE_RULES),
    liquidity,
    R.equal_weight_panel(liquidity),
)

forecast_ready = {}
volatility_ready = {}
with capture_output():
    for instrument in ALL:
        forecasts = probe.combForecast.get_all_forecasts(
            instrument, list(PROBE_RULES)
        )
        forecasts = forecasts.reindex(liquidity.index).ffill()

        observed_prices = data.daily_prices(instrument).dropna()
        count = pd.Series(
            1, index=observed_prices.index, dtype=float
        ).cumsum().reindex(liquidity.index).ffill().fillna(0.0)
        for lookback in (10, 20, 40, 80, 160, 320):
            name = f"breakout{lookback}"
            forecasts[name] = forecasts[name].where(
                count >= math.ceil(lookback / 2)
            )

        forecast_ready[instrument] = forecasts.notna()
        vol = probe.positionSize.get_average_position_at_subsystem_level(
            instrument
        )
        volatility_ready[instrument] = (
            vol.reindex(liquidity.index).ffill().notna()
        )


def eligibility_for(rule_names, flat_before=None):
    allowed = liquidity.copy()
    for instrument in ALL:
        ready = forecast_ready[instrument][list(rule_names)].all(axis=1)
        allowed[instrument] &= ready & volatility_ready[instrument]
    if flat_before is not None:
        allowed.loc[allowed.index < pd.Timestamp(flat_before)] = False
    return allowed


def matched_weights(eligibility):
    weights = R.equal_weight_panel(eligibility)
    active = weights.sum(axis=1) > 0
    assert weights.loc[active].sum(axis=1).sub(1.0).abs().max() < 1e-12
    return weights


core_eligibility = eligibility_for(CORE_RULES)
core_instruments = int((core_eligibility.sum() > 0).sum())
print(
    f"core: {core_instruments} instruments ever eligible; "
    f"{int(core_eligibility.iloc[-1].sum())} at cutoff"
)

del probe
gc.collect()

## Question 1 — Where did 70/30 go?

There are three different objects people casually call a “weight”:

1. `combForecast.get_forecast_weights()` is the configured signal budget;
2. native weighted-rule P&L is a linear, pre-buffer approximation to each
   sleeve's contribution; and
3. `accounts.portfolio()` is the actual buffered, rounded, fully costed result.

The first is not a volatility allocation.  If two equally volatile sleeves
are uncorrelated, 70/30 already implies a carry variance share of
`0.7² / (0.7² + 0.3²) = 84.5%`.

We build carry-only, trend-only, and the actual 70/30 system under exactly the
same daily market membership.  The standalone systems answer “what if this
sleeve received the whole target”; multiplying their P&L by 70% and 30% gives
a separately costed net approximation.  For cleaner structural attribution,
the native weighted-rule API supplies the relative rule/instrument weights.
Its current implementation normalises their joint sum back to one; with this
study's fixed FDM 1 and IDM 2.5 we restore that one aggregate 2.5 scale for
tracking against the actual portfolio.  We never multiply the 70/30 rule
weights a second time.

In [ ]:
core_fixed_weights = matched_weights(core_eligibility)

print("running carry-only, trend-only, and 70/30 core ...")
carry_system = native_system(
    CARRY_RULES, equal_weights(CARRY_RULES),
    core_eligibility, core_fixed_weights,
)
trend_system = native_system(
    TREND_RULES, equal_weights(TREND_RULES),
    core_eligibility, core_fixed_weights,
)
core_system = native_system(
    CORE_RULES, core_weights(0.70),
    core_eligibility, core_fixed_weights,
)
with capture_output():
    carry_frame = portfolio_frame(carry_system)
    trend_frame = portfolio_frame(trend_system)
    core_frame = portfolio_frame(core_system)
print("three core systems done")

In [ ]:
# Native linear rule attribution.  Forecast-level account curves always use
# SR costs, so gross P&L is the clean quantity here.
print("extracting native weighted-rule gross P&L ...")
with capture_output():
    weighted_rules = core_system.accounts.pandl_for_all_trading_rules(
        delayfill=True
    )
    rule_gross = pd.concat(
        {
            rule: weighted_rules[rule].percent.gross.as_ts
            for rule in CORE_RULES
        },
        axis=1,
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

# accountForecast normalises the sum of joint rule/instrument weights to one.
# Here instrument weights and forecast weights each sum to one, FDM is one,
# and IDM is fixed at 2.5, so restore that aggregate scale exactly once.
RULE_ATTRIBUTION_SCALE = 2.5
rule_gross *= RULE_ATTRIBUTION_SCALE

linear_gross_sleeves = pd.DataFrame({
    "carry": rule_gross[list(CARRY_RULES)].sum(axis=1),
    "trend": rule_gross[list(TREND_RULES)].sum(axis=1),
})

# This approximation uses actual stored costs in two complete standalone
# systems.  Separate sleeves cannot net opposing orders, so use it to describe
# net risk, not to reconcile exact costs.
standalone_net_sleeves = pd.DataFrame({
    "carry": 0.70 * carry_frame["net"],
    "trend": 0.30 * trend_frame["net"],
}).fillna(0.0)


def euler_row(components, actual, start, end, label, window):
    sample = components.loc[start:end].fillna(0.0)
    actual_sample = actual.loc[start:end].fillna(0.0)
    aligned = pd.concat(
        [sample, actual_sample.rename("actual")], axis=1
    ).fillna(0.0)
    sample = aligned[["carry", "trend"]]
    actual_sample = aligned["actual"]

    covariance = sample.cov() * TRADING_DAYS
    ones = pd.Series(1.0, index=covariance.index)
    variance = float(ones @ covariance @ ones)
    component_variance = ones * (covariance @ ones)
    risk_share = component_variance / variance
    linear = sample.sum(axis=1)
    residual = actual_sample - linear
    return {
        "basis": label,
        "window": window,
        "carry standalone vol %": (
            sample["carry"].std(ddof=1) * math.sqrt(TRADING_DAYS)
        ),
        "trend standalone vol %": (
            sample["trend"].std(ddof=1) * math.sqrt(TRADING_DAYS)
        ),
        "carry/trend correlation": sample.corr().loc["carry", "trend"],
        "carry Euler risk share": risk_share["carry"],
        "trend Euler risk share": risk_share["trend"],
        "linear vol %": linear.std(ddof=1) * math.sqrt(TRADING_DAYS),
        "actual buffered vol %": (
            actual_sample.std(ddof=1) * math.sqrt(TRADING_DAYS)
        ),
        "tracking residual vol %": (
            residual.std(ddof=1) * math.sqrt(TRADING_DAYS)
        ),
    }


RISK_WINDOWS = {
    "post-2008": (POST_2008, CUTOFF),
    "fit through 2023": (POST_2008, FIT_END),
    "revealed last 3 years": (AUDIT_START, CUTOFF),
}
risk_rows = []
for window, (start, end) in RISK_WINDOWS.items():
    risk_rows.append(euler_row(
        linear_gross_sleeves, core_frame["gross"], start, end,
        "native weighted-rule gross", window,
    ))
    risk_rows.append(euler_row(
        standalone_net_sleeves, core_frame["net"], start, end,
        "scaled standalone net", window,
    ))

risk_attribution = pd.DataFrame(risk_rows).set_index(["window", "basis"])
assert risk_attribution[[
    "carry Euler risk share", "trend Euler risk share"
]].sum(axis=1).sub(1.0).abs().max() < 1e-10
display(risk_attribution)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
risk_attribution[[
    "carry Euler risk share", "trend Euler risk share"
]].mul(100).unstack("basis").plot.bar(
    ax=axes[0], title="Forecast 70/30 becomes covariance risk shares"
)
risk_attribution[[
    "linear vol %", "actual buffered vol %", "tracking residual vol %"
]].unstack("basis").plot.bar(
    ax=axes[1], title="Linear attribution versus actual buffered portfolio"
)
axes[0].set_ylabel("percent of portfolio variance")
axes[1].set_ylabel("annualised volatility (%)")
plt.tight_layout()
plt.show()

### How to inspect the chain manually with native APIs

For one instrument, start at the forecast weights and walk down to the
buffered whole-contract target.  This is usually more educational than
reading a final Sharpe table.

In [ ]:
instrument = "SHFE_RB"
forecast_weights = core_system.combForecast.get_forecast_weights(instrument)
weighted_forecasts = (
    core_system.combForecast
    .get_weighted_forecasts_without_multiplier(instrument)
)
forecast_path = pd.DataFrame({
    "carry weighted forecast": weighted_forecasts[list(CARRY_RULES)].sum(axis=1),
    "trend weighted forecast": weighted_forecasts[list(TREND_RULES)].sum(axis=1),
    "combined after FDM/cap": (
        core_system.combForecast.get_combined_forecast(instrument)
    ),
    "subsystem position": (
        core_system.positionSize.get_subsystem_position(instrument)
    ),
    "portfolio target": (
        core_system.portfolio.get_notional_position(instrument)
    ),
    "buffered whole-contract target": (
        core_system.accounts.get_buffered_position(
            instrument, roundpositions=True
        )
    ),
})

print("Actual forecast weights for SHFE_RB:")
display(forecast_weights.tail(3))
display(forecast_path.loc["2025":].tail(10))
forecast_path[[
    "carry weighted forecast", "trend weighted forecast",
    "combined after FDM/cap",
]].loc["2023":].plot(
    figsize=(14, 5), title="SHFE_RB: forecast contributions"
)
plt.show()

Interpretation:

- forecast weights say how capped/scaled signals are averaged;
- the volatility scalar turns forecast 10 into a risk-targeted subsystem
  position;
- instrument weight and IDM create the portfolio target;
- cap, buffer, opposing signals, and integer contracts make final P&L
  nonlinear.

Use Euler covariance shares to answer “which sleeve supplies portfolio
variance?”  Use the actual complete system for Sharpe, drawdown, and costs.
Never add net forecast-level account curves and call the sum an executable
portfolio.

## Question 2 — Buffered EWMAC versus continuous Donchian

The native non-binary Donchian rule is called `breakout`.  It places price
continuously inside the rolling high/low channel and smooths the result over
roughly one quarter of the lookback.  It is not the superseded binary study's persistent
state machine.

The superseded binary study was stronger over the full common span and
2023–2026, but not over its final exact year.  More importantly, it annually
selected one of six lookbacks separately for every instrument and stayed at
forecast ±20, while fixed EWMAC mixed three speeds around mean absolute
forecast 10.  That experiment cannot isolate “binary versus continuous”.

Here the speed mapping is fixed in advance:

| EWMAC | Continuous breakout |
|---|---|
| 16/64 | 40 |
| 32/128 | 80 |
| 64/256 | 160 |

Both systems use source scalars, equal rule weights, FDM 1.5, a 10% forecast
buffer, and exactly the same daily market weights.

In [ ]:
BREAKOUT_MATCH = ("breakout40", "breakout80", "breakout160")
DONCHIAN_PAIR_RULES = TREND_RULES + BREAKOUT_MATCH
donchian_eligibility = eligibility_for(DONCHIAN_PAIR_RULES)
donchian_weights = matched_weights(donchian_eligibility)

ewmac_compare_system = native_system(
    TREND_RULES, equal_weights(TREND_RULES),
    donchian_eligibility, donchian_weights, fdm=1.5,
)
breakout_compare_system = native_system(
    BREAKOUT_MATCH, equal_weights(BREAKOUT_MATCH),
    donchian_eligibility, donchian_weights, fdm=1.5,
)

print("running matched buffered trend systems ...")
with capture_output():
    ewmac_compare = portfolio_frame(ewmac_compare_system)
    breakout_compare = portfolio_frame(breakout_compare_system)

# A separate flat launch answers the revealed-period implementation question.
flat_donchian_eligibility = eligibility_for(
    DONCHIAN_PAIR_RULES, flat_before=AUDIT_START
)
flat_donchian_weights = matched_weights(flat_donchian_eligibility)
ewmac_flat_system = native_system(
    TREND_RULES, equal_weights(TREND_RULES),
    flat_donchian_eligibility, flat_donchian_weights, fdm=1.5,
)
breakout_flat_system = native_system(
    BREAKOUT_MATCH, equal_weights(BREAKOUT_MATCH),
    flat_donchian_eligibility, flat_donchian_weights, fdm=1.5,
)
with capture_output():
    ewmac_flat = portfolio_frame(ewmac_flat_system).loc[AUDIT_START:CUTOFF]
    breakout_flat = portfolio_frame(breakout_flat_system).loc[AUDIT_START:CUTOFF]
print("matched trend comparison done")

In [ ]:
trend_rows = []
for period, start, end, frames in [
    ("post-2008 ongoing", POST_2008, CUTOFF,
     {"EWMAC": ewmac_compare, "continuous Donchian": breakout_compare}),
    ("fit through 2023", POST_2008, FIT_END,
     {"EWMAC": ewmac_compare, "continuous Donchian": breakout_compare}),
    ("revealed ongoing", AUDIT_START, CUTOFF,
     {"EWMAC": ewmac_compare, "continuous Donchian": breakout_compare}),
    ("revealed flat launch", AUDIT_START, CUTOFF,
     {"EWMAC": ewmac_flat, "continuous Donchian": breakout_flat}),
]:
    for strategy, frame in frames.items():
        trend_rows.append({
            "period": period,
            "strategy": strategy,
            **performance(frame, start, end),
        })

trend_comparison = pd.DataFrame(trend_rows).set_index(["period", "strategy"])
display(trend_comparison)

with capture_output():
    turnover = pd.Series({
        "EWMAC": ewmac_compare_system.accounts.total_portfolio_level_turnover(
            roundpositions=True
        ),
        "continuous Donchian": (
            breakout_compare_system.accounts.total_portfolio_level_turnover(
                roundpositions=True
            )
        ),
    }, name="full-history native turnover")
display(turnover.to_frame())

In [ ]:
def forecast_diagnostics(system, rule_names):
    rows = []
    with capture_output():
        for instrument in ALL:
            for rule in rule_names:
                capped = system.forecastScaleCap.get_capped_forecast(
                    instrument, rule
                ).dropna()
                if capped.empty:
                    continue
                rows.append({
                    "instrument": instrument,
                    "rule": rule,
                    "mean abs forecast": capped.abs().mean(),
                    "cap rate": capped.abs().ge(20.0 - 1e-10).mean(),
                    "observations": len(capped),
                })
    return pd.DataFrame(rows)


ewmac_forecasts = forecast_diagnostics(ewmac_compare_system, TREND_RULES)
breakout_forecasts = forecast_diagnostics(
    breakout_compare_system, BREAKOUT_MATCH
)
forecast_scale_check = pd.concat({
    "EWMAC": ewmac_forecasts.groupby("rule").apply(
        lambda x: pd.Series({
            "mean abs forecast": np.average(
                x["mean abs forecast"], weights=x["observations"]
            ),
            "mean cap rate": np.average(
                x["cap rate"], weights=x["observations"]
            ),
        }),
    ),
    "continuous Donchian": breakout_forecasts.groupby("rule").apply(
        lambda x: pd.Series({
            "mean abs forecast": np.average(
                x["mean abs forecast"], weights=x["observations"]
            ),
            "mean cap rate": np.average(
                x["cap rate"], weights=x["observations"]
            ),
        }),
    ),
})
display(forecast_scale_check)

focus = "SHFE_RB"
focus_rule = "breakout80"
native_forecast_chain = pd.concat({
    "raw": breakout_compare_system.rules.get_raw_forecast(focus, focus_rule),
    "scaled": breakout_compare_system.forecastScaleCap.get_scaled_forecast(
        focus, focus_rule
    ),
    "capped": breakout_compare_system.forecastScaleCap.get_capped_forecast(
        focus, focus_rule
    ),
    "combined": breakout_compare_system.combForecast.get_combined_forecast(
        focus
    ),
}, axis=1)
display(native_forecast_chain.loc["2025":].tail(10))
native_forecast_chain.loc["2024":].plot(
    figsize=(14, 5), title="SHFE_RB: native continuous-breakout forecast chain"
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
R.cumulative_from_zero(pd.DataFrame({
    "EWMAC": ewmac_compare.loc[POST_2008:FIT_END, "net"],
    "continuous Donchian": breakout_compare.loc[POST_2008:FIT_END, "net"],
})).plot(ax=axes[0], title="Fit-period cumulative net P&L")
R.cumulative_from_zero(pd.DataFrame({
    "EWMAC": ewmac_flat["net"],
    "continuous Donchian": breakout_flat["net"],
})).plot(ax=axes[1], title="Revealed audit, both launched flat")
plt.tight_layout()
plt.show()

saved_binary_context = pd.DataFrame({
    "binary Donchian": [1.208, 0.432, 1.321],
    "EWMAC": [0.980, 0.336, 1.427],
}, index=["binary study full", "binary study latest 3y", "binary study final year"])
print("Context only — the superseded study used a different fitted binary experiment:")
display(saved_binary_context)

## Question 3 — Which of the 28 rules pays rent beside carry/EWMAC?

The old 28-rule ladder cannot answer this: every candidate always kept all 28
rules, changed their weights, and contained no carry.  Here each row below is
a complete, fully buffered, whole-contract, stored-cost rerun.

For six families the experiment is:

```
baseline  = 70% carry + 30% EWMAC
candidate = 63% carry + 27% EWMAC + 10% one optional family
```

The baseline is rerun on the candidate's exact readiness mask, so a slow rule
cannot win merely by delaying new-market entry.  `momentum` is different:
three of its five rules are byte-for-byte the same signal parameters as the
core EWMAC rules.  Its honest experiment is therefore:

```
baseline  = 70% carry + 30% three-speed EWMAC
candidate = 70% carry + 30% five-speed momentum grid
```

This is add-one screening, not exact rule attribution.  Buffering, caps,
integer contracts, and opposing forecasts are nonlinear; subtracting two
complete portfolio P&Ls answers “what changes if I adopt this family?”

In [ ]:
def family_candidate_definition(family):
    family_rules = RULE_FAMILIES[family]
    if family == "momentum":
        active = CARRY_RULES + family_rules
        weights = {
            **{rule: 0.70 / len(CARRY_RULES) for rule in CARRY_RULES},
            **{rule: 0.30 / len(family_rules) for rule in family_rules},
        }
        change = "replace EWMAC with five-speed momentum grid"
    else:
        active = CORE_RULES + family_rules
        weights = {
            **{rule: 0.63 / len(CARRY_RULES) for rule in CARRY_RULES},
            **{rule: 0.27 / len(TREND_RULES) for rule in TREND_RULES},
            **{rule: 0.10 / len(family_rules) for rule in family_rules},
        }
        change = "add fixed 10% sleeve"
    assert abs(sum(weights.values()) - 1.0) < 1e-12
    return active, weights, change


def annual_block_sharpe_deltas(base, candidate, eligibility):
    rows = []
    for year in range(2012, 2023):
        start = pd.Timestamp(year, 7, 28)
        end = pd.Timestamp(year + 1, 7, 27)
        active_sessions = int(eligibility.any(axis=1).loc[start:end].sum())
        if active_sessions < 150:
            continue
        base_sr = sharpe(base.loc[start:end, "net"])
        candidate_sr = sharpe(candidate.loc[start:end, "net"])
        if np.isfinite(base_sr) and np.isfinite(candidate_sr):
            rows.append({
                "block": f"{year}-{year + 1}",
                "baseline Sharpe": base_sr,
                "candidate Sharpe": candidate_sr,
                "Sharpe delta": candidate_sr - base_sr,
            })
    return pd.DataFrame(rows).set_index("block")


family_frames = {}
family_blocks = {}
family_rows = []
for family in RULE_FAMILIES:
    active_rules, candidate_weights, change = family_candidate_definition(family)
    comparison_rules = tuple(dict.fromkeys(CORE_RULES + active_rules))
    eligibility = eligibility_for(comparison_rules)
    weights = matched_weights(eligibility)

    print(f"running matched baseline and {family} candidate ...")
    baseline_system = native_system(
        CORE_RULES, core_weights(0.70), eligibility, weights
    )
    candidate_system = native_system(
        active_rules, candidate_weights, eligibility, weights
    )
    with capture_output():
        baseline = portfolio_frame(baseline_system)
        candidate = portfolio_frame(candidate_system)

    blocks = annual_block_sharpe_deltas(
        baseline, candidate, eligibility
    )
    base_fit = performance(baseline, FAMILY_FIT_START, FIT_END)
    candidate_fit = performance(candidate, FAMILY_FIT_START, FIT_END)
    base_audit = performance(baseline, AUDIT_START, CUTOFF)
    candidate_audit = performance(candidate, AUDIT_START, CUTOFF)

    valid_blocks = len(blocks)
    median_delta = blocks["Sharpe delta"].median()
    positive_fraction = blocks["Sharpe delta"].gt(0).mean()
    pooled_delta = candidate_fit["net Sharpe"] - base_fit["net Sharpe"]
    double_cost_delta = (
        candidate_fit["2x-cost Sharpe"] - base_fit["2x-cost Sharpe"]
    )
    return_delta = (
        candidate_fit["net return %"] - base_fit["net return %"]
    )
    passed = bool(
        valid_blocks >= 8
        and pooled_delta >= 0.05
        and median_delta > 0.0
        and positive_fraction >= 2.0 / 3.0
        and return_delta > 0.0
        and double_cost_delta >= 0.0
    )

    family_rows.append({
        "family": family,
        "change": change,
        "rules": len(RULE_FAMILIES[family]),
        "valid blocks": valid_blocks,
        "fit baseline Sharpe": base_fit["net Sharpe"],
        "fit candidate Sharpe": candidate_fit["net Sharpe"],
        "fit Sharpe delta": pooled_delta,
        "median block Sharpe delta": median_delta,
        "positive block fraction": positive_fraction,
        "fit 2x-cost Sharpe delta": double_cost_delta,
        "fit net return delta %": return_delta,
        "fit extra cost drag %": (
            candidate_fit["cost drag %"] - base_fit["cost drag %"]
        ),
        "revealed ongoing Sharpe delta": (
            candidate_audit["net Sharpe"] - base_audit["net Sharpe"]
        ),
        "passed frozen screen": passed,
    })
    family_frames[family] = {"baseline": baseline, "candidate": candidate}
    family_blocks[family] = blocks

    del baseline_system, candidate_system
    gc.collect()

family_screen = pd.DataFrame(family_rows).set_index("family")
family_screen = family_screen.sort_values("fit Sharpe delta", ascending=False)
display(family_screen)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
family_screen[[
    "fit Sharpe delta", "revealed ongoing Sharpe delta"
]].plot.bar(ax=axes[0], title="Add-one Sharpe change")
axes[0].axhline(0.0, color="black", linewidth=0.8)
family_screen["positive block fraction"].plot.bar(
    ax=axes[1], title="Fraction of pre-2023 blocks improved"
)
axes[1].axhline(2 / 3, color="black", linestyle="--", linewidth=0.8)
family_screen[[
    "fit extra cost drag %", "fit net return delta %"
]].plot.bar(ax=axes[2], title="Cost and net-return change")
axes[2].axhline(0.0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

incremental_curves = pd.DataFrame({
    family: frames["candidate"]["net"] - frames["baseline"]["net"]
    for family, frames in family_frames.items()
})
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
R.cumulative_from_zero(
    incremental_curves.loc[FAMILY_FIT_START:FIT_END]
).plot(ax=axes[0], title="Pre-2023 incremental net P&L")
R.cumulative_from_zero(
    incremental_curves.loc[AUDIT_START:CUTOFF]
).plot(ax=axes[1], title="Revealed incremental net P&L — no selection vote")
plt.tight_layout()
plt.show()

block_delta_table = pd.concat(
    {family: blocks["Sharpe delta"] for family, blocks in family_blocks.items()},
    axis=1,
)
block_delta_table.plot.bar(
    figsize=(16, 6), title="Annual-block Sharpe change by family"
)
plt.axhline(0.0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

### Freeze the screen, then form one small candidate

Passing optional families share one 10% sleeve; passing momentum replaces the
three-speed EWMAC grid with all five speeds.  A family receives no more total
budget merely because it has more horizons.  If nothing passes, the candidate
is exactly the original core.

The final comparison uses the union of baseline and candidate readiness, then
relaunches both systems flat on 2023-07-28.  The first non-zero target must be
filled later and pay a stored native cost.

In [ ]:
PASSED_FAMILIES = family_screen.index[
    family_screen["passed frozen screen"]
].tolist()
USE_MOMENTUM_GRID = "momentum" in PASSED_FAMILIES
PASSED_OPTIONAL = [name for name in PASSED_FAMILIES if name != "momentum"]
SELECTED_TREND_RULES = (
    RULE_FAMILIES["momentum"] if USE_MOMENTUM_GRID else TREND_RULES
)


def selected_weights():
    optional_budget = 0.10 if PASSED_OPTIONAL else 0.0
    core_budget = 1.0 - optional_budget
    weights = {
        **{
            rule: core_budget * 0.70 / len(CARRY_RULES)
            for rule in CARRY_RULES
        },
        **{
            rule: core_budget * 0.30 / len(SELECTED_TREND_RULES)
            for rule in SELECTED_TREND_RULES
        },
    }
    if PASSED_OPTIONAL:
        family_budget = optional_budget / len(PASSED_OPTIONAL)
        for family in PASSED_OPTIONAL:
            for rule in RULE_FAMILIES[family]:
                weights[rule] = family_budget / len(RULE_FAMILIES[family])
    assert abs(sum(weights.values()) - 1.0) < 1e-12
    return weights


SELECTED_WEIGHTS = selected_weights()
SELECTED_RULES = tuple(SELECTED_WEIGHTS)
FINAL_COMPARISON_RULES = tuple(dict.fromkeys(CORE_RULES + SELECTED_RULES))

print("FROZEN PRE-2023 FAMILY SCREEN:", PASSED_FAMILIES or "none")
print("selected trend grid:", "momentum five-speed" if USE_MOMENTUM_GRID else "core EWMAC")
print("selected optional sleeve:", PASSED_OPTIONAL or "none")
display(pd.Series(SELECTED_WEIGHTS, name="forecast weight").to_frame())

final_eligibility = eligibility_for(FINAL_COMPARISON_RULES)
final_fixed_weights = matched_weights(final_eligibility)
baseline_final_system = native_system(
    CORE_RULES, core_weights(0.70), final_eligibility, final_fixed_weights
)
selected_final_system = native_system(
    SELECTED_RULES, SELECTED_WEIGHTS, final_eligibility, final_fixed_weights
)
with capture_output():
    baseline_final = portfolio_frame(baseline_final_system)
    selected_final = portfolio_frame(selected_final_system)

flat_final_eligibility = eligibility_for(
    FINAL_COMPARISON_RULES, flat_before=AUDIT_START
)
flat_final_weights = matched_weights(flat_final_eligibility)
baseline_final_flat_system = native_system(
    CORE_RULES, core_weights(0.70),
    flat_final_eligibility, flat_final_weights,
)
selected_final_flat_system = native_system(
    SELECTED_RULES, SELECTED_WEIGHTS,
    flat_final_eligibility, flat_final_weights,
)
with capture_output():
    baseline_final_flat = portfolio_frame(
        baseline_final_flat_system
    ).loc[AUDIT_START:CUTOFF]
    selected_final_flat = portfolio_frame(
        selected_final_flat_system
    ).loc[AUDIT_START:CUTOFF]

In [ ]:
final_rows = []
for period, start, end, frames in [
    ("pre-2023 fit", FAMILY_FIT_START, FIT_END,
     {"matched core": baseline_final, "screened candidate": selected_final}),
    ("revealed flat launch", AUDIT_START, CUTOFF,
     {"matched core": baseline_final_flat,
      "screened candidate": selected_final_flat}),
]:
    for strategy, frame in frames.items():
        final_rows.append({
            "period": period,
            "strategy": strategy,
            **performance(frame, start, end),
        })
final_comparison = pd.DataFrame(final_rows).set_index(["period", "strategy"])
display(final_comparison)


def first_costed_entry(system):
    entries = []
    with capture_output():
        for instrument in ALL:
            curve = system.accounts.pandl_for_instrument(
                instrument, delayfill=True, roundpositions=True
            )
            calculator = curve.pandl_calculator_with_costs
            held = calculator.positions.fillna(0.0)
            assert held.loc[held.index < AUDIT_START].eq(0.0).all()
            costs = calculator.costs_from_trading_in_instrument_currency_as_series()
            for fill in calculator.fills:
                fill_date = pd.Timestamp(fill.date)
                if fill_date >= AUDIT_START and abs(fill.qty) > 0:
                    cost = float(costs.reindex([fill_date]).fillna(0.0).iloc[0])
                    entries.append((fill_date, instrument, cost))
                    break
    costed = [entry for entry in entries if entry[2] < 0.0]
    assert costed
    return min(costed)


first_entry = first_costed_entry(selected_final_flat_system)
print("first delayed entry with a stored native cost:", first_entry)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
R.cumulative_from_zero(pd.DataFrame({
    "matched core": baseline_final.loc[FAMILY_FIT_START:FIT_END, "net"],
    "screened candidate": selected_final.loc[FAMILY_FIT_START:FIT_END, "net"],
})).plot(ax=axes[0], title="Frozen-screen fit comparison")
R.cumulative_from_zero(pd.DataFrame({
    "matched core": baseline_final_flat["net"],
    "screened candidate": selected_final_flat["net"],
})).plot(ax=axes[1], title="Revealed audit, both launched flat")
plt.tight_layout()
plt.show()

## What to copy into your own research

For a beginner, the useful native API surface is small:

```python
# Forecasts
system.rules.get_raw_forecast(code, rule)
system.forecastScaleCap.get_capped_forecast(code, rule)
system.combForecast.get_forecast_weights(code)
system.combForecast.get_combined_forecast(code)

# Positions
system.positionSize.get_subsystem_position(code)
system.portfolio.get_notional_position(code)
system.accounts.get_buffered_position(code, roundpositions=True)

# Fully executable P&L
curve = system.accounts.portfolio(delayfill=True, roundpositions=True)
gross = curve.percent.gross.as_ts
costs = curve.percent.costs.as_ts
net = curve.percent.as_ts
```

Use `pandl_for_all_trading_rules()` only for structural, usually gross,
attribution.  It ignores the final combined cap, portfolio buffer, inertia,
and rounding, and forecast-level costs are SR-cost approximations.  Whenever
you ask “should I change the live system?”, construct both complete systems
and compare `accounts.portfolio()` under the same dated eligibility.

Do not select the best rule/instrument pair from a heatmap.  First freeze one
family-level change, keep all its canonical horizons, compare annual blocks,
and reserve a later period that cannot vote.  New instruments should inherit
the pooled system until there is genuinely long evidence against them.

In [ ]:
net_risk = risk_attribution.loc[
    ("post-2008", "scaled standalone net")
]
fit_trend = trend_comparison.loc["fit through 2023"]
audit_trend = trend_comparison.loc["revealed flat launch"]
fit_winner = fit_trend["net Sharpe"].idxmax()
audit_winner = audit_trend["net Sharpe"].idxmax()

print("DATA-DRIVEN ANSWERS")
print(
    f"1. The 70/30 forecast budget became approximately "
    f"{100 * net_risk['carry Euler risk share']:.1f}% carry / "
    f"{100 * net_risk['trend Euler risk share']:.1f}% trend in post-2008 "
    "net covariance risk."
)
print(
    f"2. In the fair continuous comparison, {fit_winner} had the higher "
    f"pre-2023 net Sharpe ({fit_trend.loc[fit_winner, 'net Sharpe']:.3f}); "
    f"{audit_winner} had the higher flat-launch revealed Sharpe "
    f"({audit_trend.loc[audit_winner, 'net Sharpe']:.3f})."
)
print(
    "3. Families passing the frozen pre-2023 screen: "
    f"{PASSED_FAMILIES or 'none'}."
)
print(
    f"The matched core/candidate flat-launch revealed Sharpes were "
    f"{final_comparison.loc[('revealed flat launch', 'matched core'), 'net Sharpe']:.3f} "
    f"and {final_comparison.loc[('revealed flat launch', 'screened candidate'), 'net Sharpe']:.3f}."
)
print()
print(
    "Research recommendation: treat the transparent carry+EWMAC system as the "
    "benchmark. Add only a family that passes the predeclared family screen, "
    "then shadow it; do not deploy all 28 merely because their standalone "
    "portfolio diversified trend. A passing cross-sectional family still "
    "needs the stricter China-peer robustness check."
)

## What survived from the superseded experiments

The older binary-Donchian study used the same 95-history, causal-membership
idea. Over its declared common sample, binary Donchian had net Sharpe 1.208
versus 0.980 for buffered multi-speed EWMAC, and lower gross correlation with
carry (0.253 versus 0.266). Those are historical diagnostics, not a fitted
production choice; the compact lab repeats the cleaner fixed-lookback shape
test.

The v00--v05 ladder selected fast_tilt_80_20 before the holdout. Its guarded
score was 0.0825 versus -0.0493 for the Qoppac prior, and its frozen fitted
payload SHA-256 was:

    2b6f49fd7b771a3b308944ac9da2f68015ec397964d16a4731429c274210e3e0

The untouched 2023-07-28 to 2026-07-27 run reported net Sharpe 0.5183 (0.4219
with one extra observed-session lag). This short receipt preserves the result
and provenance; the six-version scaffold is intentionally no longer a runtime
dependency.